# 🔧 BrandSphere AI — 02: Data Preprocessing
**CRS AI Capstone 2025–26 | Scenario 1**

Cleans, engineers features, and prepares all 5 datasets for model training.

In [ ]:
!pip install pandas numpy scikit-learn Pillow nltk -q
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re, os, warnings
warnings.filterwarnings('ignore')
STOP_WORDS = set(stopwords.words('english'))
print('Imports OK')

## Step 1: Load Datasets

In [ ]:
np.random.seed(42)
# --- Slogan Dataset ---
slogans_list = ['Just Do It','Think Different','Open Happiness','The Best or Nothing',
                'Impossible Is Nothing','Eat Fresh','Dream Big','Make It Happen'] * 63
slogan_df = pd.DataFrame({
    'slogan': slogans_list[:500],
    'industry': np.random.choice(['Tech','Fashion','Food','Finance','Health','Education'], 500),
    'tone': np.random.choice(['Bold','Formal','Youthful','Inspirational','Luxury'], 500),
    'target_audience': np.random.choice(['B2B','B2C','Youth','Professionals'], 500),
    'word_count': np.random.randint(2, 8, 500),
    'rating': np.random.uniform(3.0, 5.0, 500).round(1)
})
# Inject some noise for cleaning demo
slogan_df.loc[np.random.choice(500, 20), 'tone'] = np.nan
slogan_df = pd.concat([slogan_df, slogan_df.iloc[:5]])  # duplicates

# --- Campaign Dataset ---
campaign_df = pd.DataFrame({
    'platform': np.random.choice(['Instagram','Facebook','Twitter/X','LinkedIn','TikTok'], 1000),
    'objective': np.random.choice(['Brand Awareness','Engagement','Conversion','Lead Generation'], 1000),
    'region': np.random.choice(['North America','Europe','Asia','Middle East','Africa'], 1000),
    'industry': np.random.choice(['Tech','Fashion','Food','Finance','Health'], 1000),
    'budget_usd': np.random.randint(500, 50000, 1000),
    'duration_days': np.random.randint(7, 90, 1000),
    'ctr_percent': np.random.uniform(0.5, 8.0, 1000).round(2),
    'roi_percent': np.random.uniform(-10, 300, 1000).round(1),
    'engagement_score': np.random.uniform(10, 100, 1000).round(1)
})
# Inject nulls
campaign_df.loc[np.random.choice(1000, 15), 'roi_percent'] = np.nan
print(f'Slogan (raw): {slogan_df.shape}, Campaign (raw): {campaign_df.shape}')

## Step 2: Data Cleaning

In [ ]:
# --- Clean Slogan Dataset ---
print('BEFORE:', slogan_df.shape, '| Nulls:', slogan_df.isnull().sum().sum(), '| Dupes:', slogan_df.duplicated().sum())
slogan_df = slogan_df.drop_duplicates()
slogan_df['tone'] = slogan_df['tone'].fillna(slogan_df['tone'].mode()[0])
slogan_df['slogan'] = slogan_df['slogan'].str.strip().str.lower()
slogan_df = slogan_df.reset_index(drop=True)
print('AFTER:', slogan_df.shape, '| Nulls:', slogan_df.isnull().sum().sum())

# --- Clean Campaign Dataset ---
print('\nCampaign BEFORE nulls:', campaign_df.isnull().sum().sum())
campaign_df['roi_percent'] = campaign_df['roi_percent'].fillna(campaign_df.groupby('platform')['roi_percent'].transform('median'))
campaign_df = campaign_df.drop_duplicates().reset_index(drop=True)
print('Campaign AFTER nulls:', campaign_df.isnull().sum().sum())

## Step 3: Feature Engineering

In [ ]:
# --- Slogan: NLP Features ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text)).lower()
    tokens = word_tokenize(text)
    return ' '.join([t for t in tokens if t not in STOP_WORDS and len(t) > 2])

slogan_df['slogan_clean'] = slogan_df['slogan'].apply(clean_text)
slogan_df['char_count'] = slogan_df['slogan'].apply(len)
slogan_df['exclamation'] = slogan_df['slogan'].apply(lambda x: int('!' in str(x)))

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=50, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(slogan_df['slogan_clean'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f'tfidf_{i}' for i in range(50)])

print('TF-IDF matrix shape:', tfidf_df.shape)
print('Top TF-IDF features:', tfidf.get_feature_names_out()[:10])

In [ ]:
# --- Campaign: Encode Categoricals + Scale Numerics ---
le = LabelEncoder()
cat_cols = ['platform', 'objective', 'region', 'industry']
for col in cat_cols:
    campaign_df[f'{col}_encoded'] = le.fit_transform(campaign_df[col])

scaler = MinMaxScaler()
num_cols = ['budget_usd', 'duration_days', 'impressions'] if 'impressions' in campaign_df.columns else ['budget_usd', 'duration_days']
scale_cols = [c for c in num_cols if c in campaign_df.columns]
campaign_df[[f'{c}_scaled' for c in scale_cols]] = scaler.fit_transform(campaign_df[scale_cols])

print('Campaign feature-engineered shape:', campaign_df.shape)
campaign_df.head(3)

## Step 4: Train / Validation / Test Split

In [ ]:
# --- Campaign Dataset Split (80/10/10) ---
feature_cols = [c for c in campaign_df.columns if '_encoded' in c or '_scaled' in c]
X = campaign_df[feature_cols]
y_ctr = campaign_df['ctr_percent']
y_roi = campaign_df['roi_percent']
y_eng = campaign_df['engagement_score']

X_train, X_temp, y_ctr_train, y_ctr_temp = train_test_split(X, y_ctr, test_size=0.2, random_state=42)
X_val, X_test, y_ctr_val, y_ctr_test = train_test_split(X_temp, y_ctr_temp, test_size=0.5, random_state=42)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')
print(f'Train ratio: {X_train.shape[0]/X.shape[0]:.0%} | Val: {X_val.shape[0]/X.shape[0]:.0%} | Test: {X_test.shape[0]/X.shape[0]:.0%}')

## Step 5: Save Cleaned Datasets

In [ ]:
os.makedirs('../datasets/cleaned', exist_ok=True)

slogan_df.to_csv('../datasets/cleaned/slogan_cleaned.csv', index=False)
campaign_df.to_csv('../datasets/cleaned/campaign_cleaned.csv', index=False)
X_train.to_csv('../datasets/cleaned/campaign_X_train.csv', index=False)
X_val.to_csv('../datasets/cleaned/campaign_X_val.csv', index=False)
X_test.to_csv('../datasets/cleaned/campaign_X_test.csv', index=False)
pd.DataFrame(y_ctr_train).to_csv('../datasets/cleaned/campaign_y_ctr_train.csv', index=False)
pd.DataFrame(y_roi).to_csv('../datasets/cleaned/campaign_y_roi.csv', index=False)

print('All cleaned datasets saved to ../datasets/cleaned/')
print('Next: Run 03_logo_model.ipynb')